# Career Knowledge Assistant — Cohere RAG

Cloud-based version of this RAG pipeline: **Cohere** for embeddings, rerank
and chat (via `ClientV2`), **FAISS + BM25 hybrid search followed by Cohere
Rerank**, structure-aware chunking with document title/section metadata,
and in-text citations. Built to mirror `core/ingestion.py`,
`core/retriever.py`, and `core/generator.py` exactly, so this notebook and
the deployable Streamlit app (`app.py`) never drift out of sync.

**Prerequisite:** copy `.env.example` to `.env` in the project root and set
a real `COHERE_API_KEY` (free trial key at
[dashboard.cohere.com/api-keys](https://dashboard.cohere.com/api-keys))
before running the cells below. The key is read server-side only — see
`core/generator.py:resolve_secret()` — and is never printed by this
notebook.

In [1]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolve the project root whether this notebook is run from the repo root
# or elsewhere, so paths and the `core` package import work on any machine.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir() and (PROJECT_ROOT.parent / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")  # no-op if .env doesn't exist yet

# Reuse the exact same engine app.py uses -- nothing in this notebook
# reimplements ingestion/retrieval/generation, it only calls core.*.
from core.ingestion import extract_text_from_pdfs, clean_document_text, build_chunks
from core.embeddings import EMBED_MODEL
from core.reranker import RERANK_MODEL
from core.retriever import build_index, retrieve
from core.generator import generate_answer, get_cohere_client, CHAT_MODEL

DATA_DIR = PROJECT_ROOT / "data"

# Raises a clear RuntimeError (and never prints the key itself) if
# COHERE_API_KEY isn't set -- see core/generator.py:resolve_secret().
cohere_client = get_cohere_client()
print(f"✓ Cohere client ready. Embed: {EMBED_MODEL} | Rerank: {RERANK_MODEL} | Chat: {CHAT_MODEL}")

records = extract_text_from_pdfs(DATA_DIR)
print(f"\nExtracted {len(records)} raw page(s) from {DATA_DIR}")

✓ Cohere client ready. Embed: embed-multilingual-v3.0 | Rerank: rerank-v3.5 | Chat: command-r-08-2024

Extracted 19 raw page(s) from C:\Users\Ahmed\Desktop\NRAG\data


In [3]:
records[0]

PageRecord(file_name='01_Resume_Writing_Best_Practices.pdf', page_number=1, text='Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter\'s 7-second skim.\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a\nperson; a beautifully designed two-column resume may impress a human but fail to parse at all.\nWhat actually changed with AI screening\n\x7f\nSemantic matching, not just keyword matching. Modern AI screening understands th

In [4]:
# clean_document_text lives in core/ingestion.py (shared with app.py) --
# this cell just demonstrates what it does to one page's raw extracted text.
sample_raw = records[0].text
sample_cleaned = clean_document_text(sample_raw)

print("--- before cleaning (first 400 chars) ---")
print(sample_raw[:400])
print("\n--- after cleaning (first 400 chars) ---")
print(sample_cleaned[:400])

--- before cleaning (first 400 chars) ---
Resume Writing Best Practices
 CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition
Target reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening
and a human recruiter's 7-second skim.
1. The Two Readers Every Resume Has
Every resume submitted online today is read twice: once by software (an Applicant Tracking System, often
layered with an AI scoring 

--- after cleaning (first 400 chars) ---
Resume Writing Best Practices
 CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition
Target reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening
and a human recruiter's 7-second skim.

1. The Two Readers Every Resume Has
Every resume submitted online today is read twice: once by software (an Applicant Tracking System, often
layered with an AI scoring


In [5]:
for r in records[:3]:
    print(f"--- {r.file_name} (page {r.page_number}) ---")
    print(clean_document_text(r.text)[:400])
    print()

--- 01_Resume_Writing_Best_Practices.pdf (page 1) ---
Resume Writing Best Practices
 CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition
Target reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening
and a human recruiter's 7-second skim.

1. The Two Readers Every Resume Has
Every resume submitted online today is read twice: once by software (an Applicant Tracking System, often
layered with an AI scoring

--- 01_Resume_Writing_Best_Practices.pdf (page 2) ---
Core Competencies / Skills -- a compact grid of 6-10 keywords pulled directly from the kind of job
descriptions you're targeting (see the companion guide on analyzing job descriptions).

Professional Experience -- reverse-chronological, with impact-driven bullets (see Section 3).

Education & Certifications -- degree, institution, graduation year (omit GPA unless above 3.5 and
you're early-career)

--- 01_Resume_Writing_Best_Practices.pdf (page 3) ---
Mirror the job description's langu

In [6]:
# build_chunks() (core/ingestion.py) re-runs extraction + cleaning, then
# splits at real section boundaries first (numbered headers) before
# falling back to size-based splitting within a section -- each chunk
# carries its document title, section heading, and page number, and its
# ID is a content hash (stable across re-runs on unchanged text).
chunks = build_chunks(DATA_DIR)
print(f"✓ Generated {len(chunks)} structure-aware chunks with rich metadata.")

✓ Generated 89 structure-aware chunks with rich metadata.


In [7]:
chunks[:3]

[Chunk(chunk_id='01_Resume_Writing_Best_Practices.pdf_p1_77b0c04da0e7f9e6', document_id='b165a6d7dbee', document_name='01_Resume_Writing_Best_Practices.pdf', document_title='Resume Writing Best Practices', section='', page_number=1, text="Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter's 7-second skim.", parent_id='b165a6d7dbee::(untitled p1)', parent_text="Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter's 7-second skim.", parent_page_start=1, parent_page_end=1),
 Chunk(chunk_id='01_Resume_Writing_Best_Practices.pdf_p1_30f4475e096d5ce4', document_id='b165a6d7dbee', document_name='01_Resume_Writing_Best_Practices.pdf', document_title='Resume Writing Bes

In [8]:
# This calls the Cohere Embed API (batched, <=96 texts/call) to embed every
# chunk, then builds an in-memory FAISS (dense) + BM25 (sparse) index --
# the exact RagIndex class app.py caches at startup. No vector DB file is
# written to disk; re-running this cell just rebuilds the same index.
index = build_index(DATA_DIR, cohere_client)
print(f"✓ Indexed {len(index)} chunks via Cohere ({EMBED_MODEL}) + FAISS + BM25.")

✓ Indexed 89 chunks via Cohere (embed-multilingual-v3.0) + FAISS + BM25.


In [9]:
def search_and_display(query: str, top_k: int = 4):
    """Pretty-print the full retrieval pipeline's results for inspection.

    retrieve() (core/retriever.py) first fuses FAISS vector search with
    BM25 keyword search via Reciprocal Rank Fusion and drops anything
    neither method is confident about, THEN passes that shortlist through
    Cohere Rerank to keep only the top_k most genuinely relevant chunks --
    this is what avoids "lost in the middle" degradation from handing the
    model a noisy top-10.
    """
    print(f"\nSearching for: '{query}'")
    print("=" * 70)

    hits = retrieve(index, query, top_k=top_k)
    if not hits:
        print("No sufficiently relevant chunks found (hybrid gate rejected everything).")
        return

    for i, hit in enumerate(hits, start=1):
        location = hit["section"] or f"Page {hit['page_number']}"
        print(
            f"Result #{i} | rerank={hit['rerank_score']} "
            f"(vector={hit['vector_sim']}% bm25={hit['bm25_score']}) "
            f"| {hit['document_title']} -- {location}"
        )
        print("-" * 70)
        print(hit["text"])
        print("=" * 70)

test_query = "How should I structure my resume bullet points to show measurable impact?"
search_and_display(test_query, top_k=4)


Searching for: 'How should I structure my resume bullet points to show measurable impact?'
Result #1 | rerank=0.8163 (vector=65.67% bm25=5.12) | Resume Writing Best Practices -- 3. Writing Bullets That Actually Land
----------------------------------------------------------------------
3. Writing Bullets That Actually Land
The single highest-leverage change most resumes need: replace duty descriptions ("Responsible for
managing a team") with evidence of impact. Use the X-Y-Z formula:
X-Y-Z formula: Accomplished [X], measured by [Y], by doing [Z].
Example: "Reduced customer churn by 18% (Y) by redesigning the onboarding email sequence (Z),
resulting in $240K in retained annual revenue (X)."
Lead every bullet with a strong action verb (led, built, reduced, automated, negotiated) -- not
"Responsible for".
Quantify wherever honestly possible: percentages, dollar amounts, time saved, scale (users, records,
requests/sec).
Result #2 | rerank=0.7441 (vector=53.94% bm25=4.74) | Resume Writing 

In [10]:
# generate_answer() (core/generator.py) is stateless -- it takes
# conversation history as a plain argument rather than owning global
# state, the same contract app.py uses with st.session_state.messages.
# This notebook just keeps a simple list and a couple of convenience
# wrappers around it.
conversation_history: list[dict] = []

def reset_conversation():
    conversation_history.clear()

def ask(question: str, top_k: int = 4) -> dict:
    result = generate_answer(index, question, conversation_history, top_k=top_k)
    conversation_history.append({"role": "user", "content": question})
    conversation_history.append({"role": "assistant", "content": result["answer"]})
    return result

In [11]:
# ====================================================
# End-to-end test
# ====================================================
reset_conversation()
test_query = "How should I structure my resume bullet points to show measurable impact?"

result = ask(test_query)

print("\n" + "=" * 70)
print("🤖 Final RAG Response (citations are inserted inline in the text):")
print("=" * 70)
print(result["answer"])

if result["cited_sources"]:
    print("\n📎 Cited sources (deduped, from Cohere's native citations -- not regex-parsed):")
    for s in result["cited_sources"]:
        print(f"   - {s['document_name']} -- {s['location']}")
elif result["sources"]:
    print("\n⚠️  The model answered without citing a specific retrieved source.")


🤖 Final RAG Response (citations are inserted inline in the text):
To structure your resume bullet points to show measurable impact, you should:

- Use the X-Y-Z formula: Accomplished [X], measured by [Y], by doing [Z]. For example: "Reduced customer churn by 18% (Y) by redesigning the onboarding email sequence (Z), resulting in $240K in retained annual revenue (X)."
- Lead every bullet with a strong action verb (e.g., led, built, reduced, automated, negotiated) rather than "Responsible for".
- Quantify wherever possible: percentages, dollar amounts, time saved, scale (users, records, requests/sec).
- Name tools and technologies in context, not as a separate list: "built an ETL pipeline in Airflow processing 2M records/day" is better than listing "Airflow" in isolation.
- Write in complete, specific sentences rather than fragments.
- Use both the acronym and the full term at least once (e.g., "Search Engine Optimization (SEO)") to match search or filter configurations.
- Ensure every b

In [12]:
# ====================================================
# Demo: conversation memory + the "I don't know" relevance gate
# ====================================================
reset_conversation()

for question in [
    "What's a good salary negotiation tactic?",           # turn 1
    "Can you give me one more tip like that?",             # turn 2 -- needs memory of turn 1
    "What is the boiling point of water on Mars?",         # turn 3 -- off-topic, should refuse
]:
    result = ask(question)
    print("\n" + "=" * 70)
    print(f"🤖 Q: {question}")
    print("=" * 70)
    print(result["answer"])
    if result["cited_sources"]:
        print("\n📎 Cited:", ", ".join(f"{s['document_name']} ({s['location']})" for s in result["cited_sources"]))


🤖 Q: What's a good salary negotiation tactic?
Before you begin any salary negotiation, it's important to prepare. Research the market rate from multiple sources, such as salary aggregation sites, industry surveys, and your professional network. Triangulate these sources rather than trusting one number. [Source: Salary Negotiation Playbook, Section: 1. Preparation: Know Your Number Before Any Conversation]

Consider your specific variables, such as location, company size/stage, years of experience, and specialized/high-demand skills. Decide on your target, walk-away minimum, and an ideal number above your target. Understand the full compensation package, including base salary, bonus structure, equity/stock, retirement matching, health benefits, remote/flexibility policy, learning budget, and PTO. [Source: Salary Negotiation Playbook, Section: 1. Preparation: Know Your Number Before Any Conversation]

During the negotiation, let the other party anchor first if possible. If asked for you